In [7]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import numpy as np
import torch
torch.set_num_threads(1)
from braindecode.models import EEGNeX

X = np.load("data/fenetres.npy")
y = np.load("data/y.npy")
groupes = np.load("data/groupes.npy", allow_pickle=True)
print(X.shape, y.shape, groupes.shape)

(1206, 129, 200) (1206,) (1206,)


In [8]:
std_fen = np.median(X[:, :128, :].std(axis=2), axis=1)
plat = std_fen < 1e-9

X, y, groupes = X[~plat], y[~plat], groupes[~plat]
X = (X * 1e6).astype(np.float32)
y = y.astype(np.float32)

print(X.shape, np.isnan(X).sum())   # attendu : (1143, 129, 200) 0  

(1143, 129, 200) 0


In [9]:
model = EEGNeX(n_chans=129, n_outputs=1, n_times=200, sfreq=100)
model.eval()
with torch.no_grad():
    out = model(torch.from_numpy(X[:8]))
print(out.shape)   # attendu : torch.Size([8, 1])

torch.Size([8, 1])


In [10]:
from sklearn.model_selection import GroupKFold
from torch.utils.data import TensorDataset, DataLoader

gkf = GroupKFold(n_splits=5)
train_idx, val_idx = next(gkf.split(X, y, groupes))

X_tr, y_tr = X[train_idx], y[train_idx]
X_va, y_va = X[val_idx], y[val_idx]

# la référence à battre sur CE pli : prédire la moyenne du train
rmse_moyenne = np.sqrt(np.mean((y_va - y_tr.mean())**2))
print(X_tr.shape, X_va.shape, rmse_moyenne)


(903, 129, 200) (240, 129, 200) 0.3353075


In [12]:
train_ds = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr))
val_ds   = TensorDataset(torch.from_numpy(X_va), torch.from_numpy(y_va))

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False)

xb, yb = next(iter(train_loader))
print(xb.shape, yb.shape)   # attendu : (64, 129, 200) (64,)

torch.Size([64, 129, 200]) torch.Size([64])


In [13]:
torch.manual_seed(0)
model = EEGNeX(n_chans=129, n_outputs=1, n_times=200, sfreq=100)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
loss_fn = torch.nn.MSELoss()
y_mean = float(y_tr.mean())   # le réseau part de ~0, on lui donne la moyenne comme point de départ

for epoch in range(10):
    # --- entraînement ---
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()                  # remet les gradients à zéro
        pred = model(xb).squeeze(1) + y_mean   # (64,1) -> (64,)
        loss = loss_fn(pred, yb)               # erreur quadratique moyenne du batch
        loss.backward()                        # calcule les gradients
        optimizer.step()                       # met à jour les poids

    # --- validation ---
    model.eval()
    erreurs = []
    with torch.no_grad():                      # pas de gradients ici
        for xb, yb in val_loader:
            pred = model(xb).squeeze(1) + y_mean
            erreurs.append((pred - yb) ** 2)
    rmse_val = torch.cat(erreurs).mean().sqrt().item()
    print(f"epoch {epoch+1:2d} | loss train {loss.item():.4f} | RMSE val {rmse_val:.4f}")

epoch  1 | loss train 0.1954 | RMSE val 0.3403
epoch  2 | loss train 0.0270 | RMSE val 0.3564
epoch  3 | loss train 0.1226 | RMSE val 0.3365
epoch  4 | loss train 0.0571 | RMSE val 0.3680
epoch  5 | loss train 0.1876 | RMSE val 0.3778
epoch  6 | loss train 0.2791 | RMSE val 0.3379
epoch  7 | loss train 0.1993 | RMSE val 0.3413
epoch  8 | loss train 0.1463 | RMSE val 0.3550
epoch  9 | loss train 0.0547 | RMSE val 0.3466
epoch 10 | loss train 0.1175 | RMSE val 0.3540


## Réseau de neurones sur le signal brut (EEGNeX)

Pour comparer avec les features, j'ai entraîné EEGNeX (braindecode / PyTorch) directement sur les fenêtres brutes de 2 s, (129 canaux × 200 points), mises en µV.

L'évaluation porte sur un seul pli du `GroupKFold` : 903 fenêtres pour l'entraînement, 240 pour la validation, avec des sujets différents dans chaque ensemble. L'entraînement dure 10 epochs avec AdamW (lr = 1e-3) et une perte MSE.

La référence est prédire la moyenne du train : RMSE = 0.335 s sur ce pli.

Le réseau reste entre 0.337 et 0.378 s de RMSE en validation, sans passer sous cette référence. C'est le même niveau que Ridge et XGBoost sur les features. L'entraînement prend environ 21 min pour un seul pli.

Le réseau ne fait pas mieux que les modèles à features. Avec 19 sujets, le temps de réaction dépend surtout de la personne, et peu de ce que l'on peut lire dans 2 s d'EEG. Le test n'a qu'un pli, une graine et aucun réglage, donc il ne prouve pas qu'aucun réseau ne marcherait.